# 05 — Cross-Source Data Fusion: BGS + USGS

This notebook fuses three complementary data sources to build a holistic picture of global critical-mineral supply chains:

| Source | File | Contents |
|--------|------|----------|
| **BGS Production** | `bgs_critical_minerals_production.csv` | Country-level mine/refinery production by year |
| **BGS Trade** | `bgs_critical_minerals_trade.csv` | Country-level import/export flows by year |
| **USGS MCS 2024 World** | `usgs_mcs_data/2024/world.zip` | 2022/2023 production estimates + reserves per country |

### Why cross-source fusion?
No single database covers all minerals, all countries, and all time horizons equally well.  
BGS provides long time-series (often back to the 1900s); USGS provides the most recent estimates and reserve data.  
By aligning them on country and commodity we can:
- Fill temporal gaps
- Cross-validate figures
- Build richer "mineral health" scorecards that combine production, trade, and reserve context

**Output artefacts**: mineral health scorecard DataFrame, production-share heatmap, and a radar chart comparing top-producing countries across all key minerals.

## 1 · Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Data paths ─────────────────────────────────────────────────────────────────
DATA_DIR  = Path('../data')
BGS_DIR   = DATA_DIR / 'bgs_data'
USGS_DIR  = DATA_DIR / 'usgs_mcs_data' / '2024'

print(f'DATA_DIR  : {DATA_DIR.resolve()}')
print(f'BGS_DIR   : {BGS_DIR.resolve()}')
print(f'USGS_DIR  : {USGS_DIR.resolve()}')

## 2 · Load BGS Data

We load both the production and trade CSVs, coerce `year` and `quantity` to numeric types,
and retain only records where `statistic_type` indicates production (for the production file)
or import/export (for the trade file).

In [ ]:
# ── Load ───────────────────────────────────────────────────────────────────────
bgs_prod_raw  = pd.read_csv(BGS_DIR / 'bgs_critical_minerals_production.csv', low_memory=False)
bgs_trade_raw = pd.read_csv(BGS_DIR / 'bgs_critical_minerals_trade.csv',      low_memory=False)

def clean_bgs(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce year/quantity to numeric and drop rows with neither."""
    df = df.copy()
    df['year']     = pd.to_numeric(df['year'],     errors='coerce')
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df = df.dropna(subset=['year', 'quantity'])
    df['year']     = df['year'].astype(int)
    df['quantity'] = df['quantity'].astype(float)
    # Normalise text columns
    for col in ['commodity', 'statistic_type', 'country']:
        if col in df.columns:
            df[col] = df[col].str.strip().str.lower()
    return df

bgs_prod  = clean_bgs(bgs_prod_raw)
bgs_trade = clean_bgs(bgs_trade_raw)

# Filter production rows only from the production file
bgs_prod = bgs_prod[bgs_prod['statistic_type'] == 'production']

# Separate imports and exports from the trade file
bgs_imports = bgs_trade[bgs_trade['statistic_type'] == 'imports'].copy()
bgs_exports = bgs_trade[bgs_trade['statistic_type'] == 'exports'].copy()

print(f'BGS production rows  : {len(bgs_prod):>8,}')
print(f'BGS import rows      : {len(bgs_imports):>8,}')
print(f'BGS export rows      : {len(bgs_exports):>8,}')
print(f'\nUnique commodities (production): {bgs_prod["commodity"].nunique()}')
print(f'Unique countries  (production): {bgs_prod["country"].nunique()}')
print(f'Year range (production): {bgs_prod["year"].min()} – {bgs_prod["year"].max()}')

In [ ]:
# Quick overview of available commodities in production data
bgs_prod['commodity'].value_counts().head(20)

## 3 · Load USGS World Production from ZIP

The USGS MCS 2024 `world.zip` archive contains one CSV per commodity named
`mcs2024-<code>_world.csv` (e.g., `mcs2024-lithi_world.csv` → commodity code `lithi`).
Each file records estimated 2022 and 2023 production plus reserve figures for every country.

In [ ]:
def extract_commodity_code(filename: str) -> str:
    """mcs2024-lithi_world.csv  ->  lithi"""
    stem = Path(filename).stem.lower()          # mcs2024-lithi_world
    parts = stem.split('-', 1)                  # ['mcs2024', 'lithi_world']
    if len(parts) == 2:
        return parts[1].replace('_world', '')   # lithi
    return stem

usgs_world: dict[str, pd.DataFrame] = {}

with zipfile.ZipFile(USGS_DIR / 'world.zip') as zf:
    csv_names = sorted(
        n for n in zf.namelist()
        if n.lower().endswith('_world.csv')
    )
    for name in csv_names:
        code = extract_commodity_code(name)
        with zf.open(name) as f:
            df = pd.read_csv(f, low_memory=False)
            # Keep only useful columns; rename for consistency
            df.columns = [c.strip() for c in df.columns]
            # Identify production columns (vary between files: Prod_t_2022 or Prod_t_est_2022)
            prod_cols = [c for c in df.columns if 'prod' in c.lower() and ('2022' in c or '2023' in c)]
            reserve_cols = [c for c in df.columns if 'reserve' in c.lower() and 'note' not in c.lower()]
            keep = ['Country'] + prod_cols + reserve_cols
            keep = [c for c in keep if c in df.columns]
            df = df[keep].copy()
            # Coerce numeric
            for col in prod_cols + reserve_cols:
                df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce')
            df = df.dropna(subset=['Country'])
            df['Country'] = df['Country'].str.strip()
            df['commodity_code'] = code
            usgs_world[code] = df

print(f'USGS commodity codes loaded: {len(usgs_world)}')
print('Sample codes:', list(usgs_world.keys())[:15])

In [ ]:
# Inspect columns for a few key commodities
for code in ['lithi', 'cobal', 'coppe', 'nicke', 'graph']:
    if code in usgs_world:
        print(f'{code:6s}: {usgs_world[code].columns.tolist()}')

## 4 · Mineral Health Scorecard

For each of nine key critical minerals we compute, per country:

| Metric | Source | Definition |
|--------|--------|------------|
| `production` | BGS (most recent 5-year avg) | Average annual mine production (tonnes) |
| `exports` | BGS (most recent 5-year avg) | Average annual export quantity |
| `imports` | BGS (most recent 5-year avg) | Average annual import quantity |
| `global_share_pct` | BGS production | Country share of world total (%) |
| `self_sufficiency_pct` | BGS prod + trade | `production / (production + imports - exports)` capped at 200% |

**Key minerals mapped to BGS commodity names:**

In [ ]:
# ── Commodity mapping: display name -> BGS commodity string ───────────────────
MINERAL_MAP = {
    'Lithium'      : 'lithium minerals',
    'Cobalt'       : 'cobalt mine',
    'Nickel'       : 'nickel mine',
    'Graphite'     : 'graphite',
    'Copper'       : 'copper mine',
    'Manganese'    : 'manganese ore',
    'Rare Earths'  : 'rare earth minerals',
    'Tungsten'     : 'tungsten',
    'Vanadium'     : 'vanadium',
}

RECENT_YEARS = 5  # average over last N years of data

def recent_avg(df, commodity, recent_n=RECENT_YEARS):
    """Return per-country average quantity over the most recent `recent_n` years."""
    sub = df[df['commodity'] == commodity]
    if sub.empty:
        return pd.Series(dtype=float)
    max_year = sub['year'].max()
    sub = sub[sub['year'] > max_year - recent_n]
    return sub.groupby('country')['quantity'].mean()

scorecard_rows = []

for mineral_name, bgs_commodity in MINERAL_MAP.items():
    prod_s   = recent_avg(bgs_prod,    bgs_commodity)
    exp_s    = recent_avg(bgs_exports, bgs_commodity)
    imp_s    = recent_avg(bgs_imports, bgs_commodity)

    if prod_s.empty:
        print(f'  [SKIP] No production data for: {bgs_commodity}')
        continue

    # Align on country index
    all_countries = prod_s.index.union(exp_s.index).union(imp_s.index)
    prod_a = prod_s.reindex(all_countries, fill_value=0.0)
    exp_a  = exp_s.reindex(all_countries,  fill_value=0.0)
    imp_a  = imp_s.reindex(all_countries,  fill_value=0.0)

    world_total = prod_a.sum()
    global_share = (prod_a / world_total * 100) if world_total > 0 else prod_a * 0

    # Self-sufficiency = production / apparent consumption
    # apparent_consumption = production + imports - exports  (clamped to > 0)
    apparent_consumption = (prod_a + imp_a - exp_a).clip(lower=1e-9)
    self_suff = (prod_a / apparent_consumption * 100).clip(upper=200.0)

    for country in all_countries:
        scorecard_rows.append({
            'mineral'             : mineral_name,
            'country'             : country,
            'production'          : prod_a[country],
            'exports'             : exp_a[country],
            'imports'             : imp_a[country],
            'global_share_pct'    : global_share[country],
            'self_sufficiency_pct': self_suff[country],
        })

scorecard = pd.DataFrame(scorecard_rows)
scorecard = scorecard[scorecard['production'] > 0].copy()  # drop non-producers

print(f'Scorecard rows  : {len(scorecard):,}')
print(f'Minerals covered: {scorecard["mineral"].nunique()}')
print(f'Countries covered: {scorecard["country"].nunique()}')
scorecard.head(10)

## 5 · Production-Share Heatmap

We pivot the scorecard to a **country × mineral** matrix of global-share percentages,
then filter to countries that have a share > 5 % in at least one mineral.
The colour scale (YlOrRd) highlights concentration risk at a glance.

In [ ]:
# ── Pivot: country × mineral -> global_share_pct ──────────────────────────────
pivot = scorecard.pivot_table(
    index='country', columns='mineral', values='global_share_pct', aggfunc='sum', fill_value=0
)

# Filter: keep countries with at least one mineral share > 5 %
significant = pivot[(pivot > 5).any(axis=1)]
# Sort rows by total share descending
significant = significant.loc[significant.sum(axis=1).sort_values(ascending=False).index]

print(f'Countries shown: {len(significant)}')

fig_heatmap = px.imshow(
    significant,
    color_continuous_scale='YlOrRd',
    zmin=0,
    zmax=100,
    aspect='auto',
    labels=dict(x='Mineral', y='Country', color='Global Share (%)'),
    title='Global Production Share by Country and Mineral (%, 5-year average)<br><sup>Countries with at least one share > 5% shown</sup>',
    text_auto='.1f',
)

fig_heatmap.update_layout(
    height=max(400, 28 * len(significant)),
    width=900,
    xaxis_tickangle=-35,
    coloraxis_colorbar=dict(title='Share (%)', ticksuffix='%'),
    margin=dict(l=160, r=60, t=80, b=80),
    font=dict(size=12),
)

fig_heatmap.update_traces(textfont_size=10)
fig_heatmap.show()

## 6 · Radar Chart — Top 5 Producers Across All Minerals

A radar (spider) chart shows each of the top 5 producing countries as a polygon
whose vertices are the country's global-share percentage for each mineral.
This makes it easy to see *breadth* (many minerals) versus *specialisation* (one or two dominant minerals).

In [ ]:
# ── Identify top 5 countries by total production share across all minerals ────
top5_countries = (
    scorecard
    .groupby('country')['global_share_pct']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
    .tolist()
)

print('Top 5 producing countries (by summed share across minerals):')
for rank, c in enumerate(top5_countries, 1):
    total_share = scorecard[scorecard['country'] == c]['global_share_pct'].sum()
    print(f'  {rank}. {c.title()} — {total_share:.1f}% aggregate share')

minerals_list = sorted(scorecard['mineral'].unique().tolist())

In [ ]:
# ── Build radar chart ─────────────────────────────────────────────────────────
# Plotly Scatterpolar requires the first and last theta/r values to be the same
# to close the polygon.

PALETTE = px.colors.qualitative.Bold

fig_radar = go.Figure()

for i, country in enumerate(top5_countries):
    country_data = scorecard[scorecard['country'] == country].set_index('mineral')
    shares = [
        country_data.loc[m, 'global_share_pct'] if m in country_data.index else 0.0
        for m in minerals_list
    ]
    # Close the polygon
    r_vals     = shares + [shares[0]]
    theta_vals = minerals_list + [minerals_list[0]]

    fig_radar.add_trace(go.Scatterpolar(
        r          = r_vals,
        theta      = theta_vals,
        fill       = 'toself',
        name       = country.title(),
        line       = dict(color=PALETTE[i % len(PALETTE)], width=2),
        fillcolor  = PALETTE[i % len(PALETTE)],
        opacity    = 0.25,
        hovertemplate=(
            f'<b>{country.title()}</b><br>'
            'Mineral: %{theta}<br>'
            'Share: %{r:.1f}%<extra></extra>'
        ),
    ))

fig_radar.update_layout(
    title=dict(
        text=(
            'Top 5 Countries — Production Share Across Critical Minerals<br>'
            '<sup>5-year average global share (%)</sup>'
        ),
        x=0.5,
    ),
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100],
            ticksuffix='%',
            showline=False,
            gridcolor='rgba(200,200,200,0.5)',
        ),
        angularaxis=dict(
            tickfont=dict(size=12),
            gridcolor='rgba(200,200,200,0.5)',
        ),
        bgcolor='rgba(245,245,245,0.6)',
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.15,
        xanchor='center',
        x=0.5,
    ),
    height=600,
    width=700,
    margin=dict(t=100, b=100),
    paper_bgcolor='white',
)

fig_radar.show()

## 7 · Self-Sufficiency Overview

A supplementary grouped-bar chart shows self-sufficiency scores for the top 5 producers
across all minerals.  A score of 100 % means a country produces exactly what it consumes;
> 100 % indicates a net exporter; < 100 % indicates import dependence.

In [ ]:
top5_suff = scorecard[scorecard['country'].isin(top5_countries)].copy()
top5_suff['country_label'] = top5_suff['country'].str.title()

fig_bar = px.bar(
    top5_suff,
    x='mineral',
    y='self_sufficiency_pct',
    color='country_label',
    barmode='group',
    color_discrete_sequence=px.colors.qualitative.Bold,
    labels={
        'mineral'             : 'Mineral',
        'self_sufficiency_pct': 'Self-Sufficiency (%)',
        'country_label'       : 'Country',
    },
    title='Self-Sufficiency by Mineral for Top 5 Producing Countries<br><sup>Capped at 200% — above 100% = net exporter</sup>',
)

fig_bar.add_hline(
    y=100,
    line_dash='dash',
    line_color='black',
    annotation_text='100% (self-sufficient)',
    annotation_position='top right',
    annotation_font_size=11,
)

fig_bar.update_layout(
    height=500,
    xaxis_tickangle=-25,
    yaxis=dict(range=[0, 210]),
    legend_title='Country',
    margin=dict(t=80, b=80),
)

fig_bar.show()

## 8 · Summary

This notebook demonstrated a complete cross-source fusion pipeline:

1. **BGS production** (long time-series) cleaned and aggregated to 5-year recent averages.
2. **BGS trade** (imports + exports) used to compute net trade balances and self-sufficiency.
3. **USGS MCS 2024 world ZIP** loaded programmatically for reserve and near-term production context.
4. A **mineral health scorecard** DataFrame was built, summarising production, trade, global share, and self-sufficiency per country per mineral.
5. A **YlOrRd heatmap** revealed which countries dominate which minerals (concentration risk).
6. A **radar chart** compared the breadth vs. depth of the top 5 producers across all nine key minerals.
7. A **grouped bar chart** surfaced self-sufficiency gaps — critical for supply-chain resilience analysis.

**Next steps**:
- Integrate USGS reserve data to add a *reserve lifetime* metric (reserves / current production).
- Apply time-series forecasting on BGS production trends.
- Incorporate geopolitical risk scores (HHI, WGI) per country for supply-chain vulnerability ranking.